# 🎯 Análise de Centralidade - Game of Thrones
## Quem é matematicamente o personagem mais importante?

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [2]:
df = pd.read_csv("../datasets/interacoes.csv")
df_direct = df#[df["tipo_interacao"] == "single"]
df_grouped = df_direct.groupby(['falante', 'ouvinte']).size().reset_index(name='weight')
G = nx.from_pandas_edgelist(df_grouped, 'falante', 'ouvinte', edge_attr='weight', create_using=nx.Graph())
print(f"✅ Grafo: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

✅ Grafo: 274 nós, 1278 arestas


## 1️⃣ Centralidade de Grau

In [3]:
degree_cent = nx.degree_centrality(G)
top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_degree, columns=['Personagem', 'Score'])

,Personagem,Score
0,TYRION LANNISTER,0.234432
1,JON SNOW,0.223443
2,DAENERYS TARGARYEN,0.183150
3,CERSEI LANNISTER,0.179487
4,SANSA STARK,0.172161
5,JAIME LANNISTER,0.168498
6,DAVOS SEAWORTH,0.161172
7,THEON GREYJOY,0.150183
8,ARYA STARK,0.146520
9,BRAN STARK,0.146520


## 2️⃣ Centralidade de Intermediação

In [4]:
betweenness_cent = nx.betweenness_centrality(G, weight='weight')
top_betweenness = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_betweenness, columns=['Personagem', 'Score'])

,Personagem,Score
0,ARYA STARK,0.217358
1,DAVOS SEAWORTH,0.175922
2,BRONN,0.143591
3,BRAN STARK,0.121036
4,DAENERYS TARGARYEN,0.115410
5,THEON GREYJOY,0.088973
6,BRIENNE DE TARTH,0.080941
7,TORMUND GIANTSBANE,0.073017
8,SANDOR CLEGANE,0.067030
9,JORAH MORMONT,0.065000


## 3️⃣ PageRank

In [5]:
pagerank = nx.pagerank(G, weight='weight', max_iter=100, tol=1e-06)
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_pagerank, columns=['Personagem', 'Score'])

,Personagem,Score
0,TYRION LANNISTER,0.054125
1,JON SNOW,0.043405
2,DAENERYS TARGARYEN,0.033629
3,CERSEI LANNISTER,0.027440
4,SANSA STARK,0.025450
5,JAIME LANNISTER,0.024568
6,ARYA STARK,0.024369
7,SAMWELL TARLY,0.021771
8,THEON GREYJOY,0.020720
9,DAVOS SEAWORTH,0.018591


## 4️⃣ Closeness Centrality

In [6]:
closeness_centrality = nx.closeness_centrality(G, distance='weight')
top_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_closeness, columns=['Personagem', 'Score'])

,Personagem,Score
0,ARYA STARK,0.195169
1,BRAN STARK,0.189305
2,DAVOS SEAWORTH,0.187142
3,BRONN,0.185308
4,TORMUND GIANTSBANE,0.182826
5,BRIENNE DE TARTH,0.182148
6,LANCEL LANNISTER,0.180542
7,THEON GREYJOY,0.179881
8,PODRICK PAYNE,0.179750
9,DAENERYS TARGARYEN,0.179356


## 5️⃣ Weighted Degree

In [7]:
weighted_degree = dict(G.degree(weight='weight'))
top_weighted = sorted(weighted_degree.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_weighted, columns=['Personagem', 'Interações'])

,Personagem,Interações
0,TYRION LANNISTER,2607
1,JON SNOW,1784
2,DAENERYS TARGARYEN,1447
3,CERSEI LANNISTER,1211
4,JAIME LANNISTER,1130
5,SANSA STARK,1125
6,ARYA STARK,1046
7,SAMWELL TARLY,835
8,DAVOS SEAWORTH,812
9,THEON GREYJOY,701


## 🏆 Ranking Consolidado
**Pesos:** PageRank(30%), Betweenness(25%), Degree(25%), Closeness(20%)

In [8]:
personagens = list(degree_cent.keys())
scaler = MinMaxScaler()

norm_pr = scaler.fit_transform(np.array([pagerank[p] for p in personagens]).reshape(-1,1)).flatten()
norm_bt = scaler.fit_transform(np.array([betweenness_cent[p] for p in personagens]).reshape(-1,1)).flatten()
norm_dc = scaler.fit_transform(np.array([degree_cent[p] for p in personagens]).reshape(-1,1)).flatten()
norm_cc = scaler.fit_transform(np.array([closeness_centrality[p] for p in personagens]).reshape(-1,1)).flatten()

scores = {p: norm_pr[i]*0.3 + norm_bt[i]*0.25 + norm_dc[i]*0.25 + norm_cc[i]*0.2 for i,p in enumerate(personagens)}
top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:20]

df_final = pd.DataFrame([
    (p, scores[p], pagerank[p], betweenness_cent[p], degree_cent[p], closeness_centrality[p]) 
    for p,_ in top
], columns=["Personagem","Score","PageRank","Betweenness","Degree","Closeness"])

df_final

,Personagem,Score,PageRank,Betweenness,Degree,Closeness
0,TYRION LANNISTER,0.789069,0.054125,0.053683,0.234432,0.173041
1,ARYA STARK,0.738080,0.024369,0.217358,0.146520,0.195169
2,JON SNOW,0.697271,0.043405,0.039828,0.223443,0.169227
3,DAENERYS TARGARYEN,0.696173,0.033629,0.115410,0.183150,0.179356
4,DAVOS SEAWORTH,0.665708,0.018591,0.175922,0.161172,0.187142
5,CERSEI LANNISTER,0.593728,0.027440,0.063133,0.179487,0.175764
6,BRONN,0.577085,0.013995,0.143591,0.139194,0.185308
7,BRAN STARK,0.574003,0.015928,0.121036,0.146520,0.189305
8,THEON GREYJOY,0.558278,0.020720,0.088973,0.150183,0.179881
9,SANSA STARK,0.547079,0.025450,0.042781,0.172161,0.171711
